# Modelo de Momento Óptimo y Detección de Anomalías (Share Velocity)

Este notebook implementa la lógica de **Share of Wallet** y **Share Velocity** para detectar el momento óptimo de actuación comercial y disparar alarmas reactivas ante fugas inminentes.

### Definiciones clave:
1.  **Share of Wallet (SOW)**: % de la capacidad de compra del cliente que captamos nosotros.
2.  **Share Velocity**: Cambio mensual en el SOW (la "película" de la tendencia).
3.  **Alarma Reactiva**: Alerta inmediata cuando la velocidad cae más de un 5% en un solo mes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración de visualización
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Parámetros de negocio
THRESHOLD_ALERTA_FUGA = -0.05  # Caída del 5% mensual
DATA_PATH = 'data/master_commodities.csv'

## 1. Carga y Preparación de Datos

In [ ]:
def load_and_clean_data(path):
    df = pd.read_csv(path)
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    
    # Solo ventas netas (excluir devoluciones)
    df = df[df['es_devolucion'] == 0].copy()
    
    # Asegurar que el potencial es numérico
    df['Potencial_EUR_anual'] = pd.to_numeric(df['Potencial_EUR_anual'], errors='coerce')
    
    return df

df = load_and_clean_data(DATA_PATH)
print(f"Dataset cargado: {len(df)} filas")
df.head()

## 2. Cálculo del Share of Wallet y Velocidad

Agrupamos mensualmente y calculamos el acumulado de los últimos 12 meses (rolling) para obtener el Share estable.

In [ ]:
def calculate_metrics(df):
    # 1. Agregar ventas por (Cliente, Familia, Mes)
    df['any_mes'] = df['Fecha'].dt.to_period('M')
    
    monthly = df.groupby(['Id_Cliente', 'Familia_Potencial', 'any_mes']).agg({
        'Valores_H': 'sum',
        'Potencial_EUR_anual': 'first'
    }).reset_index()
    
    monthly = monthly.sort_values(['Id_Cliente', 'Familia_Potencial', 'any_mes'])
    
    # 2. Ventas acumuladas 12 meses (Rolling)
    monthly['ventas_12m'] = monthly.groupby(['Id_Cliente', 'Familia_Potencial'])['Valores_H'].transform(
        lambda x: x.rolling(12, min_periods=1).sum()
    )
    
    # 3. Share of Wallet
    monthly['share'] = (monthly['ventas_12m'] / monthly['Potencial_EUR_anual']).clip(0, 1.5) # Dejamos margen por si superan potencial
    
    # 4. VELOCIDAD DEL SHARE (Cambio mensual)
    monthly['share_velocity'] = monthly.groupby(['Id_Cliente', 'Familia_Potencial'])['share'].diff().fillna(0)
    
    # 5. ALARMA REACTIVA (Detección de Anomalías)
    monthly['alarma_fuga'] = monthly['share_velocity'] < THRESHOLD_ALERTA_FUGA
    
    return monthly

metrics = calculate_metrics(df)
print(f"Métricas calculadas para {metrics['Id_Cliente'].nunique()} clientes")

## 3. Identificación de Clientes con Alarmas Activas

Buscamos los casos más críticos del último mes disponible.

In [ ]:
last_month = metrics['any_mes'].max()
critical_cases = metrics[(metrics['any_mes'] == last_month) & (metrics['alarma_fuga'] == True)]

print(f"Alertas detectadas en el mes {last_month}: {len(critical_cases)}")
critical_cases.sort_values('share_velocity').head(10)

## 4. Visualización del Momento Óptimo

Función para graficar la evolución de un cliente y entender su comportamiento.

In [ ]:
def plot_client_behavior(client_id, family):
    data = metrics[(metrics['Id_Cliente'] == client_id) & (metrics['Familia_Potencial'] == family)].copy()
    data['Fecha'] = data['any_mes'].dt.to_timestamp()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, height_ratios=[2, 1])
    
    # Plot 1: Share of Wallet
    ax1.plot(data['Fecha'], data['share'], marker='o', color='blue', label='Share of Wallet (SOW)')
    ax1.set_ylabel('Share (0-1)')
    ax1.set_title(f"Evolución Cliente {client_id} - Familia {family}")
    ax1.legend(loc='upper left')
    
    # Plot 2: Velocity
    colors = ['red' if v < THRESHOLD_ALERTA_FUGA else ('green' if v > 0 else 'gray') for v in data['share_velocity']]
    ax2.bar(data['Fecha'], data['share_velocity'], width=20, color=colors, alpha=0.6, label='Velocidad del Share')
    ax2.axhline(THRESHOLD_ALERTA_FUGA, color='red', linestyle='--', label='Umbral Alarma (-5%)')
    ax2.set_ylabel('Velocidad (Delta Mensual)')
    ax2.legend(loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Seleccionar un cliente con alerta para visualizar
if not critical_cases.empty:
    target_client = critical_cases.iloc[0]['Id_Cliente']
    target_family = critical_cases.iloc[0]['Familia_Potencial']
    plot_client_behavior(target_client, target_family)
else:
    # Si no hay alertas en el último mes, mostramos uno cualquiera con historial
    sample = metrics[metrics['share_velocity'] != 0].iloc[0]
    plot_client_behavior(sample['Id_Cliente'], sample['Familia_Potencial'])

## 5. Resumen de Acciones de Negocio

| Señal | Situación | Acción Recomendada |
| :--- | :--- | :--- |
| **Barras Verdes (Vel. > 0)** | Enamoramiento | **Fidelizar**: Ofrecer programas de puntos o cross-sell. |
| **Barras Grises (Vel. ~ 0)** | Estabilidad | **Mantenimiento**: Recordatorios automáticos de reposición. |
| **Barras Rojas (Vel. < -5%)** | **Fuga Inminente** | **Intervención Directa**: Llamada del delegado con oferta agresiva para frenar el desvío a la competencia. |